<a href="https://colab.research.google.com/github/Amankumar1456/Dofus_Pulpit/blob/main/00_prepare_banking_database_and_policy_docs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Erica-style Banking Assistant
## Database and Small Policy Documents

This notebook prepares only the shared foundation needed before App 1:

1. Creates a synthetic retail-banking SQLite database.
2. Seeds customers, accounts, cards and transactions.
3. Creates real banking operations for transaction search and card controls.
4. Writes four small Markdown policy files.
5. Reads those files directly into one prompt string.

There is **no RAG, embedding model, vector database, chunking or retrieval index** in this notebook.

The policy files are intentionally small enough to load directly into the model prompt.



## Output

```text
banking-memory-lab/
├── data/
│   └── banking_demo.db
├── knowledge/
│   ├── transaction_search_policy.md
│   ├── card_lock_policy.md
│   ├── unrecognized_transaction_policy.md
│   └── privacy_and_session_policy.md
└── 00_prepare_banking_database_and_policy_docs.ipynb
```

The notebook works in Google Colab, JupyterLab and VS Code notebooks.


In [ ]:

# SQLite is included with Python.
# pandas is used only to display the seeded data clearly.

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pandas") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "pandas"]
    )

print("Notebook dependencies are ready.")


Notebook dependencies are ready.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

from __future__ import annotations

import json
import os
import sqlite3
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any
from uuid import uuid4

import pandas as pd


IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/My Drive/FDE sessions/Combined Batch/Week 3")
else:
    # Start Jupyter or VS Code from the project folder.
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
KNOWLEDGE_DIR = PROJECT_ROOT / "knowledge"

DATA_DIR.mkdir(parents=True, exist_ok=True)
KNOWLEDGE_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DIR / "banking_demo.db"

print("Running in Colab:", IN_COLAB)
print("Project root:", PROJECT_ROOT.resolve())
print("Database path:", DB_PATH.resolve())


Running in Colab: True
Project root: /content/drive/My Drive/FDE sessions/Combined Batch/Week 3
Database path: /content/drive/My Drive/FDE sessions/Combined Batch/Week 3/data/banking_demo.db


## 1. Create four small policy documents

In [ ]:

POLICY_DOCUMENTS = {
    "transaction_search_policy.md": """
# Transaction Search Policy

- Search only accounts belonging to the authenticated customer.
- Return no more than ten recent transactions.
- Show transaction ID, merchant, amount, date and posting status.
- When the customer says "that transaction", resolve it only from the current conversation thread.
- If the reference is unclear, ask the customer to select the transaction again.
- Transaction details must come from the database tool, not from model memory.
""",

    "card_lock_policy.md": """
# Temporary Card Lock Policy

Before locking a card:

1. Confirm the card belongs to the authenticated customer.
2. Identify the card through a trusted transaction or explicit card selection.
3. Tell the customer the card's last four digits.
4. Obtain explicit confirmation.
5. Use a request ID so retries do not create duplicate actions.

A temporary lock does not reverse a posted transaction.
Never claim the card was locked until the card tool succeeds.
""",

    "unrecognized_transaction_policy.md": """
# Unrecognized Transaction Policy

- Retrieve the exact transaction before advising the customer.
- Summarize merchant, amount, date, status and card last four digits.
- Offer a temporary card lock as a safety option.
- Do not lock the card without explicit customer confirmation.
- Create a specialist handoff when the customer confirms the transaction is not theirs or asks to report fraud.
- This educational assistant does not make final fraud or refund decisions.
""",

    "privacy_and_session_policy.md": """
# Privacy and Session Policy

- Conversation state must be isolated by tenant, customer and thread.
- Recommended key: chat:{tenant_id}:{customer_id}:{thread_id}
- Never use one global key such as chat:default.
- Never return another customer's selected transaction, card or messages.
- Never expose complete card numbers, passwords or tokens.
- Hot conversation memory must have a defined TTL.
""",
}

for filename, content in POLICY_DOCUMENTS.items():
    path = KNOWLEDGE_DIR / filename
    path.write_text(content.strip() + "\n", encoding="utf-8")

print(f"Created {len(POLICY_DOCUMENTS)} policy documents:")
for path in sorted(KNOWLEDGE_DIR.glob("*.md")):
    print("-", path.name)


Created 4 policy documents:
- card_lock_policy.md
- privacy_and_session_policy.md
- transaction_search_policy.md
- unrecognized_transaction_policy.md


## 2. Read the policy documents directly into the prompt

In [ ]:

def load_policy_documents(
    policy_directory: Path = KNOWLEDGE_DIR,
) -> str:
    blocks = []

    for path in sorted(policy_directory.glob("*.md")):
        content = path.read_text(encoding="utf-8").strip()

        blocks.append(
            f"FILE: {path.name}\n"
            f"{content}"
        )

    if not blocks:
        raise RuntimeError(
            f"No Markdown policy files found in {policy_directory}"
        )

    return "\n\n---\n\n".join(blocks)


POLICY_CONTEXT = load_policy_documents()

print(POLICY_CONTEXT)


FILE: card_lock_policy.md
# Temporary Card Lock Policy

Before locking a card:

1. Confirm the card belongs to the authenticated customer.
2. Identify the card through a trusted transaction or explicit card selection.
3. Tell the customer the card's last four digits.
4. Obtain explicit confirmation.
5. Use a request ID so retries do not create duplicate actions.

A temporary lock does not reverse a posted transaction.
Never claim the card was locked until the card tool succeeds.

---

FILE: privacy_and_session_policy.md
# Privacy and Session Policy

- Conversation state must be isolated by tenant, customer and thread.
- Recommended key: chat:{tenant_id}:{customer_id}:{thread_id}
- Never use one global key such as chat:default.
- Never return another customer's selected transaction, card or messages.
- Never expose complete card numbers, passwords or tokens.
- Hot conversation memory must have a defined TTL.

---

FILE: transaction_search_policy.md
# Transaction Search Policy

- Search 

In [ ]:

SYSTEM_PROMPT = f'''
You are a retail-banking support assistant.

Use:
1. the authenticated customer's current conversation;
2. authoritative facts returned by tools;
3. the policy documents below.

Rules:
- Never invent transaction, card or handoff status.
- Never expose another customer's information.
- Never claim an action succeeded before its tool succeeds.
- Ask for explicit confirmation before locking a card.
- Keep answers short and operational.

POLICY DOCUMENTS
================
{POLICY_CONTEXT}
'''

print(SYSTEM_PROMPT)



You are a retail-banking support assistant.

Use:
1. the authenticated customer's current conversation;
2. authoritative facts returned by tools;
3. the policy documents below.

Rules:
- Never invent transaction, card or handoff status.
- Never expose another customer's information.
- Never claim an action succeeded before its tool succeeds.
- Ask for explicit confirmation before locking a card.
- Keep answers short and operational.

POLICY DOCUMENTS
FILE: card_lock_policy.md
# Temporary Card Lock Policy

Before locking a card:

1. Confirm the card belongs to the authenticated customer.
2. Identify the card through a trusted transaction or explicit card selection.
3. Tell the customer the card's last four digits.
4. Obtain explicit confirmation.
5. Use a request ID so retries do not create duplicate actions.

A temporary lock does not reverse a posted transaction.
Never claim the card was locked until the card tool succeeds.

---

FILE: privacy_and_session_policy.md
# Privacy and Sess


This is simple prompt injection of trusted local documents:

```text
small Markdown files
        ↓
read with pathlib
        ↓
combined into POLICY_CONTEXT
        ↓
inserted into SYSTEM_PROMPT
```

Later, when the policy collection becomes too large for one prompt, this can be replaced with RAG. That is deliberately outside this workshop.


## 3. Create the SQLite banking database

In [ ]:

SCHEMA_SQL = '''
PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS customers (
    customer_id TEXT PRIMARY KEY,
    full_name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    created_at TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS accounts (
    account_id TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL,
    account_type TEXT NOT NULL,
    status TEXT NOT NULL,
    available_balance_cents INTEGER NOT NULL,
    currency TEXT NOT NULL,
    created_at TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE IF NOT EXISTS cards (
    card_id TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL,
    account_id TEXT NOT NULL,
    card_last4 TEXT NOT NULL,
    card_type TEXT NOT NULL,
    status TEXT NOT NULL,
    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (account_id) REFERENCES accounts(account_id)
);

CREATE TABLE IF NOT EXISTS transactions (
    transaction_id TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL,
    account_id TEXT NOT NULL,
    card_id TEXT NOT NULL,
    merchant_name TEXT NOT NULL,
    amount_cents INTEGER NOT NULL,
    currency TEXT NOT NULL,
    category TEXT NOT NULL,
    transaction_date TEXT NOT NULL,
    status TEXT NOT NULL,
    created_at TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (account_id) REFERENCES accounts(account_id),
    FOREIGN KEY (card_id) REFERENCES cards(card_id)
);

CREATE TABLE IF NOT EXISTS card_actions (
    action_id TEXT PRIMARY KEY,
    request_id TEXT NOT NULL UNIQUE,
    customer_id TEXT NOT NULL,
    card_id TEXT NOT NULL,
    action_type TEXT NOT NULL,
    reason TEXT NOT NULL,
    resulting_status TEXT NOT NULL,
    created_at TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (card_id) REFERENCES cards(card_id)
);

CREATE TABLE IF NOT EXISTS specialist_handoffs (
    handoff_id TEXT PRIMARY KEY,
    request_id TEXT NOT NULL UNIQUE,
    customer_id TEXT NOT NULL,
    transaction_id TEXT NOT NULL,
    queue_name TEXT NOT NULL,
    reason TEXT NOT NULL,
    conversation_summary TEXT NOT NULL,
    status TEXT NOT NULL,
    created_at TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (transaction_id) REFERENCES transactions(transaction_id)
);

CREATE INDEX IF NOT EXISTS idx_transactions_customer_date
ON transactions(customer_id, transaction_date DESC);

CREATE INDEX IF NOT EXISTS idx_transactions_customer_merchant
ON transactions(customer_id, merchant_name);

CREATE INDEX IF NOT EXISTS idx_cards_customer
ON cards(customer_id);
'''

with sqlite3.connect(DB_PATH) as conn:
    conn.executescript(SCHEMA_SQL)

print("Database schema created.")


Database schema created.



## 4. Seed synthetic customers and transactions

We seed 60 customers so the later instructor demo can simulate many independent banking customers.

The Locust script can read customers directly from this SQLite database. A separate `locust_users.json` file is not needed.


In [ ]:

FIRST_NAMES = [
    "Avery", "Jordan", "Taylor", "Morgan", "Riley",
    "Casey", "Cameron", "Quinn", "Parker", "Reese",
]

LAST_NAMES = [
    "Adams", "Brooks", "Carter", "Diaz", "Evans", "Foster",
]

MERCHANTS = [
    ("FreshMart", 8645, "groceries"),
    ("MetroFuel", 5420, "transport"),
    ("StreamFlix", 1599, "subscription"),
    ("UrbanTrail", 4999, "retail"),
    ("CityCab", 2375, "transport"),
]

BASE_TIME = datetime(2026, 7, 20, 10, 0, tzinfo=timezone.utc)


def seed_database(customer_count: int = 60) -> None:
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("PRAGMA foreign_keys = ON")

        for table in [
            "specialist_handoffs",
            "card_actions",
            "transactions",
            "cards",
            "accounts",
            "customers",
        ]:
            conn.execute(f"DELETE FROM {table}")

        for index in range(1, customer_count + 1):
            customer_id = f"CUS-{index:04d}"
            account_id = f"ACC-{index:04d}"
            card_id = f"CARD-{index:04d}"
            card_last4 = f"{1000 + index:04d}"

            first_name = FIRST_NAMES[(index - 1) % len(FIRST_NAMES)]
            last_name = LAST_NAMES[(index - 1) % len(LAST_NAMES)]
            full_name = f"{first_name} {last_name}"

            conn.execute(
                '''
                INSERT INTO customers (
                    customer_id, full_name, email, created_at
                )
                VALUES (?, ?, ?, ?)
                ''',
                (
                    customer_id,
                    full_name,
                    f"{first_name.lower()}.{last_name.lower()}{index}@example.com",
                    BASE_TIME.isoformat(),
                ),
            )

            conn.execute(
                '''
                INSERT INTO accounts (
                    account_id, customer_id, account_type, status,
                    available_balance_cents, currency, created_at
                )
                VALUES (?, ?, ?, ?, ?, ?, ?)
                ''',
                (
                    account_id,
                    customer_id,
                    "checking",
                    "active",
                    600_000 + (index * 1_000),
                    "USD",
                    BASE_TIME.isoformat(),
                ),
            )

            conn.execute(
                '''
                INSERT INTO cards (
                    card_id, customer_id, account_id, card_last4,
                    card_type, status, created_at, updated_at
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                ''',
                (
                    card_id,
                    customer_id,
                    account_id,
                    card_last4,
                    "debit",
                    "active",
                    BASE_TIME.isoformat(),
                    BASE_TIME.isoformat(),
                ),
            )

            for position, (merchant, base_amount, category) in enumerate(
                MERCHANTS,
                start=1,
            ):
                transaction_id = f"TXN-{index:04d}-{position:02d}"
                amount_cents = base_amount + index

                transaction_time = BASE_TIME - timedelta(
                    days=position - 1,
                    minutes=index,
                )

                conn.execute(
                    '''
                    INSERT INTO transactions (
                        transaction_id, customer_id, account_id, card_id,
                        merchant_name, amount_cents, currency, category,
                        transaction_date, status, created_at
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                    ''',
                    (
                        transaction_id,
                        customer_id,
                        account_id,
                        card_id,
                        merchant,
                        amount_cents,
                        "USD",
                        category,
                        transaction_time.isoformat(),
                        "posted",
                        BASE_TIME.isoformat(),
                    ),
                )


seed_database(customer_count=60)
print("Synthetic database seeded.")


Synthetic database seeded.


In [ ]:

with sqlite3.connect(DB_PATH) as conn:
    table_counts = pd.read_sql_query(
        '''
        SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM customers
        UNION ALL
        SELECT 'accounts', COUNT(*) FROM accounts
        UNION ALL
        SELECT 'cards', COUNT(*) FROM cards
        UNION ALL
        SELECT 'transactions', COUNT(*) FROM transactions
        UNION ALL
        SELECT 'card_actions', COUNT(*) FROM card_actions
        UNION ALL
        SELECT 'specialist_handoffs', COUNT(*) FROM specialist_handoffs
        ''',
        conn,
    )

display(table_counts)


,table_name,row_count
0,customers,60
1,accounts,60
2,cards,60
3,transactions,300
4,card_actions,0
5,specialist_handoffs,0


In [ ]:

with sqlite3.connect(DB_PATH) as conn:
    sample_transactions = pd.read_sql_query(
        '''
        SELECT
            t.transaction_id,
            t.customer_id,
            t.merchant_name,
            ROUND(t.amount_cents / 100.0, 2) AS amount,
            t.currency,
            t.status,
            c.card_last4
        FROM transactions AS t
        JOIN cards AS c
          ON c.card_id = t.card_id
        WHERE t.customer_id = 'CUS-0001'
        ORDER BY t.transaction_date DESC
        ''',
        conn,
    )

display(sample_transactions)


,transaction_id,customer_id,merchant_name,amount,currency,status,card_last4
0,TXN-0001-01,CUS-0001,FreshMart,86.46,USD,posted,1001
1,TXN-0001-02,CUS-0001,MetroFuel,54.21,USD,posted,1001
2,TXN-0001-03,CUS-0001,StreamFlix,16.00,USD,posted,1001
3,TXN-0001-04,CUS-0001,UrbanTrail,50.00,USD,posted,1001
4,TXN-0001-05,CUS-0001,CityCab,23.76,USD,posted,1001


## 5. Create the database operations used by the later apps

In [ ]:

def search_transactions(
    customer_id: str,
    account_id: str | None = None,
    merchant_name: str | None = None,
    limit: int = 5,
) -> list[dict[str, Any]]:
    safe_limit = max(1, min(limit, 10))

    query = '''
        SELECT
            t.transaction_id,
            t.customer_id,
            t.account_id,
            t.card_id,
            c.card_last4,
            t.merchant_name,
            t.amount_cents,
            t.currency,
            t.category,
            t.transaction_date,
            t.status
        FROM transactions AS t
        JOIN cards AS c
          ON c.card_id = t.card_id
        WHERE t.customer_id = ?
    '''

    parameters: list[Any] = [customer_id]

    if account_id is not None:
        query += " AND t.account_id = ?"
        parameters.append(account_id)

    if merchant_name is not None:
        query += " AND LOWER(t.merchant_name) LIKE LOWER(?)"
        parameters.append(f"%{merchant_name}%")

    query += " ORDER BY t.transaction_date DESC LIMIT ?"
    parameters.append(safe_limit)

    with sqlite3.connect(DB_PATH) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(query, parameters).fetchall()

    results = []

    for row in rows:
        record = dict(row)
        record["amount"] = record.pop("amount_cents") / 100
        results.append(record)

    return results


def get_transaction(
    customer_id: str,
    transaction_id: str,
) -> dict[str, Any] | None:
    with sqlite3.connect(DB_PATH) as conn:
        conn.row_factory = sqlite3.Row

        row = conn.execute(
            '''
            SELECT
                t.transaction_id,
                t.customer_id,
                t.account_id,
                t.card_id,
                c.card_last4,
                t.merchant_name,
                t.amount_cents,
                t.currency,
                t.category,
                t.transaction_date,
                t.status
            FROM transactions AS t
            JOIN cards AS c
              ON c.card_id = t.card_id
            WHERE t.customer_id = ?
              AND t.transaction_id = ?
            ''',
            (customer_id, transaction_id),
        ).fetchone()

    if row is None:
        return None

    record = dict(row)
    record["amount"] = record.pop("amount_cents") / 100
    return record


def get_card_status(
    customer_id: str,
    card_id: str,
) -> dict[str, Any] | None:
    with sqlite3.connect(DB_PATH) as conn:
        conn.row_factory = sqlite3.Row

        row = conn.execute(
            '''
            SELECT
                card_id,
                customer_id,
                account_id,
                card_last4,
                card_type,
                status,
                updated_at
            FROM cards
            WHERE customer_id = ?
              AND card_id = ?
            ''',
            (customer_id, card_id),
        ).fetchone()

    return dict(row) if row is not None else None


In [ ]:

def lock_card(
    request_id: str,
    customer_id: str,
    card_id: str,
    reason: str,
) -> dict[str, Any]:
    now = datetime.now(timezone.utc).isoformat()

    with sqlite3.connect(DB_PATH) as conn:
        conn.row_factory = sqlite3.Row
        conn.execute("PRAGMA foreign_keys = ON")

        existing_action = conn.execute(
            '''
            SELECT *
            FROM card_actions
            WHERE request_id = ?
            ''',
            (request_id,),
        ).fetchone()

        if existing_action is not None:
            return {
                "status": "already_processed",
                "action": dict(existing_action),
            }

        card = conn.execute(
            '''
            SELECT *
            FROM cards
            WHERE customer_id = ?
              AND card_id = ?
            ''',
            (customer_id, card_id),
        ).fetchone()

        if card is None:
            return {
                "status": "failed",
                "reason": "Card was not found for this customer.",
            }

        action_id = f"ACT-{uuid4().hex[:10].upper()}"
        resulting_status = "temporarily_locked"

        conn.execute(
            '''
            UPDATE cards
            SET status = ?, updated_at = ?
            WHERE customer_id = ?
              AND card_id = ?
            ''',
            (resulting_status, now, customer_id, card_id),
        )

        conn.execute(
            '''
            INSERT INTO card_actions (
                action_id, request_id, customer_id, card_id,
                action_type, reason, resulting_status, created_at
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            ''',
            (
                action_id,
                request_id,
                customer_id,
                card_id,
                "temporary_lock",
                reason,
                resulting_status,
                now,
            ),
        )

    return {
        "status": "success",
        "action_id": action_id,
        "card_id": card_id,
        "card_last4": card["card_last4"],
        "resulting_status": resulting_status,
    }


def create_specialist_handoff(
    request_id: str,
    customer_id: str,
    transaction_id: str,
    reason: str,
    conversation_summary: str,
    queue_name: str = "fraud-review",
) -> dict[str, Any]:
    now = datetime.now(timezone.utc).isoformat()

    with sqlite3.connect(DB_PATH) as conn:
        conn.row_factory = sqlite3.Row
        conn.execute("PRAGMA foreign_keys = ON")

        existing_handoff = conn.execute(
            '''
            SELECT *
            FROM specialist_handoffs
            WHERE request_id = ?
            ''',
            (request_id,),
        ).fetchone()

        if existing_handoff is not None:
            return {
                "status": "already_processed",
                "handoff": dict(existing_handoff),
            }

        transaction = conn.execute(
            '''
            SELECT transaction_id
            FROM transactions
            WHERE customer_id = ?
              AND transaction_id = ?
            ''',
            (customer_id, transaction_id),
        ).fetchone()

        if transaction is None:
            return {
                "status": "failed",
                "reason": "Transaction was not found for this customer.",
            }

        handoff_id = f"HOF-{uuid4().hex[:10].upper()}"

        conn.execute(
            '''
            INSERT INTO specialist_handoffs (
                handoff_id, request_id, customer_id, transaction_id,
                queue_name, reason, conversation_summary, status, created_at
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''',
            (
                handoff_id,
                request_id,
                customer_id,
                transaction_id,
                queue_name,
                reason,
                conversation_summary,
                "queued",
                now,
            ),
        )

    return {
        "status": "success",
        "handoff_id": handoff_id,
        "queue_name": queue_name,
        "handoff_status": "queued",
    }


### Tool smoke tests

In [ ]:

urbantrail_transactions = search_transactions(
    customer_id="CUS-0001",
    merchant_name="UrbanTrail",
)

selected_transaction = urbantrail_transactions[0]

print("Selected transaction:")
print(json.dumps(selected_transaction, indent=2))

print("\nCard status:")
print(
    json.dumps(
        get_card_status(
            customer_id="CUS-0001",
            card_id=selected_transaction["card_id"],
        ),
        indent=2,
    )
)

print("\nCross-customer authorization test:")
print(
    get_transaction(
        customer_id="CUS-0002",
        transaction_id=selected_transaction["transaction_id"],
    )
)


Selected transaction:
{
  "transaction_id": "TXN-0001-04",
  "customer_id": "CUS-0001",
  "account_id": "ACC-0001",
  "card_id": "CARD-0001",
  "card_last4": "1001",
  "merchant_name": "UrbanTrail",
  "currency": "USD",
  "category": "retail",
  "transaction_date": "2026-07-17T09:59:00+00:00",
  "status": "posted",
  "amount": 50.0
}

Card status:
{
  "card_id": "CARD-0001",
  "customer_id": "CUS-0001",
  "account_id": "ACC-0001",
  "card_last4": "1001",
  "card_type": "debit",
  "status": "active",
  "updated_at": "2026-07-20T10:00:00+00:00"
}

Cross-customer authorization test:
None


In [ ]:

# Check idempotency, then restore the card so the prepared data remains clean.

test_request_id = "setup-lock-test-001"

first_result = lock_card(
    request_id=test_request_id,
    customer_id="CUS-0001",
    card_id="CARD-0001",
    reason="Notebook setup test",
)

second_result = lock_card(
    request_id=test_request_id,
    customer_id="CUS-0001",
    card_id="CARD-0001",
    reason="Notebook setup test",
)

print("First call:", first_result["status"])
print("Repeated call:", second_result["status"])

with sqlite3.connect(DB_PATH) as conn:
    conn.execute(
        '''
        UPDATE cards
        SET status = 'active',
            updated_at = ?
        WHERE card_id = 'CARD-0001'
        ''',
        (BASE_TIME.isoformat(),),
    )

    conn.execute(
        "DELETE FROM card_actions WHERE request_id = ?",
        (test_request_id,),
    )

print("Demo card restored to active.")


First call: success
Repeated call: already_processed
Demo card restored to active.


## 6. How the later Locust script gets 50 users

In [ ]:

def load_locust_customers(
    limit: int = 50,
) -> list[dict[str, Any]]:
    with sqlite3.connect(DB_PATH) as conn:
        conn.row_factory = sqlite3.Row

        rows = conn.execute(
            '''
            SELECT
                c.customer_id,
                a.account_id,
                cd.card_id,
                cd.card_last4,
                t.transaction_id AS target_transaction_id,
                t.merchant_name AS target_merchant,
                t.amount_cents AS target_amount_cents
            FROM customers AS c
            JOIN accounts AS a
              ON a.customer_id = c.customer_id
            JOIN cards AS cd
              ON cd.customer_id = c.customer_id
            JOIN transactions AS t
              ON t.customer_id = c.customer_id
             AND t.merchant_name = 'UrbanTrail'
            ORDER BY c.customer_id
            LIMIT ?
            ''',
            (limit,),
        ).fetchall()

    customers = []

    for row in rows:
        record = dict(row)
        record["target_amount"] = (
            record.pop("target_amount_cents") / 100
        )
        record["thread_id"] = (
            f"transaction-review-"
            f"{record['customer_id'].lower()}"
        )
        customers.append(record)

    return customers


locust_customers = load_locust_customers(limit=50)

print("Customers available for Locust:", len(locust_customers))
print(json.dumps(locust_customers[0], indent=2))


Customers available for Locust: 50
{
  "customer_id": "CUS-0001",
  "account_id": "ACC-0001",
  "card_id": "CARD-0001",
  "card_last4": "1001",
  "target_transaction_id": "TXN-0001-04",
  "target_merchant": "UrbanTrail",
  "target_amount": 50.0,
  "thread_id": "transaction-review-cus-0001"
}



The later Locust file can call the same SQL query during startup, or use customer IDs such as:

```text
CUS-0001
CUS-0002
...
CUS-0050
```

A separate JSON fixture adds another file to manage without teaching anything important in this session, so it has been removed.


## 7. Final validation

In [ ]:

with sqlite3.connect(DB_PATH) as conn:
    customer_count = conn.execute(
        "SELECT COUNT(*) FROM customers"
    ).fetchone()[0]

    transaction_count = conn.execute(
        "SELECT COUNT(*) FROM transactions"
    ).fetchone()[0]

checks = {
    "database_exists": DB_PATH.exists(),
    "customer_count_is_60": customer_count == 60,
    "transaction_count_is_300": transaction_count == 300,
    "policy_document_count_is_4": (
        len(list(KNOWLEDGE_DIR.glob("*.md"))) == 4
    ),
    "policy_context_loaded": len(POLICY_CONTEXT) > 100,
    "locust_customers_available": len(locust_customers) == 50,
}

display(
    pd.DataFrame(
        [
            {"check": name, "passed": passed}
            for name, passed in checks.items()
        ]
    )
)

assert all(checks.values()), "One or more setup checks failed."

print("\n[PASS] Banking database and policy documents are ready.")
print("Database:", DB_PATH)
print("Policies:", KNOWLEDGE_DIR)


,check,passed
0,database_exists,True
1,customer_count_is_60,True
2,transaction_count_is_300,True
3,policy_document_count_is_4,True
4,policy_context_loaded,True
5,locust_customers_available,True



[PASS] Banking database and policy documents are ready.
Database: /content/drive/My Drive/FDE sessions/Combined Batch/Week 3/data/banking_demo.db
Policies: /content/drive/My Drive/FDE sessions/Combined Batch/Week 3/knowledge



## Ready for App 1

App 1 will use:

- one FastAPI worker;
- process-local chat history;
- the SQLite tools prepared here;
- the four policy files loaded directly into the system prompt.

The customer journey will be:

1. Show recent transactions.
2. Select the UrbanTrail transaction.
3. Say the transaction is not recognized.
4. Ask to lock the card used for **that transaction**.
5. Confirm the lock.
6. Verify that the SQLite card status changed.

App 2 will run the same journey with four workers and expose process-local memory fragmentation.
